# 共享单车骑行数据清洗与基础统计

- 数据：`题5_共享单车骑行_202401.csv`，2024 年 1 月，238,834 行、13 列
- 流程：去重 → 归一类别 → 转数值 → 补缺删缺 → 标记异常 → 基础聚合统计
- 输出：`output/` 下的清洗结果、统计表与报告


In [ ]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 120)
pd.set_option("display.float_format", lambda value: f"{value:,.1f}")

RAW_CSV = Path("题5_共享单车骑行_202401.csv")
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)


## 读取原始数据

先按文本读入，保留空串，避免 pandas 提前推断类型。

In [ ]:
df = pd.read_csv(RAW_CSV, dtype=str, keep_default_na=False)
print(f"原始数据: {len(df)} 行 × {df.shape[1]} 列")
df.head()


## 1. 去重

以业务主键 `ride_id` 判断重复，同一笔订单只保留首次出现的记录。

In [ ]:
before = len(df)
print("重复 ride_id 涉及行数:", int(df.duplicated(subset=["ride_id"], keep=False).sum()))

df = df.drop_duplicates(subset=["ride_id"], keep="first").reset_index(drop=True)
print(f"去重: {before} -> {len(df)} 行")


## 2. 归一类别

站点名称先去掉 NBSP 和首尾空白，再把连续空白压成一个空格；
类别列统一小写，并把同义写法映射到规范取值。


In [ ]:
def clean_text(series):
    return (
        series.astype("string")
        .str.replace("\xa0", " ", regex=False)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )


for col in ["start_station_name", "end_station_name"]:
    before = df[col].astype("string")
    df[col] = clean_text(df[col])
    print(f"{col} 规范化: {int((before != df[col]).sum())} 行")

for col in ["start_station_id", "end_station_id"]:
    df[col] = clean_text(df[col])

synonyms = {
    "rideable_type": {
        "classic": "classic_bike",
        "classic bike": "classic_bike",
        "electric": "electric_bike",
        "ebike": "electric_bike",
        "e-bike": "electric_bike",
        "electric bike": "electric_bike",
    },
    "member_casual": {
        "subscriber": "member",
        "registered": "member",
        "annual": "member",
        "customer": "casual",
        "non-member": "casual",
        "nonmember": "casual",
    },
}

for col in ["rideable_type", "member_casual"]:
    df[col] = clean_text(df[col]).str.lower().replace(synonyms[col])
    print(col, sorted(df[col].dropna().unique()))


## 3. 转数值

时间戳转 datetime、经纬度转 float，并派生骑行时长 `duration_sec`。

In [ ]:
for col in ["started_at", "ended_at"]:
    df[col] = pd.to_datetime(df[col], errors="coerce")

df["duration_sec"] = (df["ended_at"] - df["started_at"]).dt.total_seconds()

for col in ["start_lat", "start_lng", "end_lat", "end_lng"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("时间范围:", df["started_at"].min(), "~", df["ended_at"].max())
print("骑行时长(秒):", df["duration_sec"].min(), "~", df["duration_sec"].max())


## 4. 补缺删缺

关键字段（时间、坐标）缺失的记录无法参与统计，删除整行；
站点名称缺失填为“未知站点”，保留记录并显式标注。


In [ ]:
before = len(df)

for col in ["start_station_name", "end_station_name", "start_station_id", "end_station_id"]:
    df[col] = df[col].replace("", pd.NA)

missing = df.isna().sum()
print("缺失诊断:")
print(missing[missing > 0].to_string())

df = df.dropna(
    subset=["ride_id", "started_at", "ended_at", "start_lat", "start_lng", "end_lat", "end_lng"]
).reset_index(drop=True)

for col in ["start_station_name", "end_station_name"]:
    df[col] = df[col].fillna("未知站点")

print(f"补缺删缺: {before} -> {len(df)} 行")


## 5. 标记异常

先按业务阈值标记，再剔除无法解释为真实骑行的硬异常；不足 60 秒的骑行只标记、不删除。


In [ ]:
duration = df["duration_sec"]

df["flag_nonpositive_duration"] = duration <= 0
df["flag_short_duration"] = (duration > 0) & (duration < 60)
df["flag_long_duration"] = duration > 6 * 3600
df["flag_over_24h"] = duration > 24 * 3600
df["flag_bad_start_coord"] = ~(
    df["start_lat"].between(38.4, 39.4) & df["start_lng"].between(-77.6, -76.6)
)
df["flag_bad_end_coord"] = ~(
    df["end_lat"].between(38.4, 39.4) & df["end_lng"].between(-77.6, -76.6)
)
df["flag_out_of_month"] = ~df["started_at"].between("2024-01-01", "2024-02-01")

rules = {
    "flag_nonpositive_duration": "骑行时长<=0",
    "flag_short_duration": "骑行时长<60秒",
    "flag_long_duration": "骑行时长>6小时",
    "flag_over_24h": "骑行时长>24小时",
    "flag_bad_start_coord": "起点坐标越界",
    "flag_bad_end_coord": "终点坐标越界",
    "flag_out_of_month": "不在2024年1月",
}

reason = pd.Series("", index=df.index, dtype="string")
for flag, label in rules.items():
    print(f"{label}: {int(df[flag].sum())} 行")
    reason = reason + df[flag].map({True: f"{label}；", False: ""})

df["anomaly_reason"] = reason.str.rstrip("；")
df["is_anomaly"] = df["anomaly_reason"] != ""

hard_flags = [
    "flag_nonpositive_duration",
    "flag_over_24h",
    "flag_bad_start_coord",
    "flag_bad_end_coord",
    "flag_out_of_month",
]
hard_mask = df[hard_flags].any(axis=1)
cleaned = df.loc[~hard_mask].reset_index(drop=True)

print(
    f"标记 {int(df['is_anomaly'].sum())} 行；"
    f"剔除硬异常 {int(hard_mask.sum())} 行；保留 {len(cleaned)} 行"
)


## 保存清洗结果

In [ ]:
df.to_csv(OUTPUT_DIR / "rides_marked.csv", index=False, encoding="utf-8-sig")
cleaned.to_csv(OUTPUT_DIR / "rides_clean.csv", index=False, encoding="utf-8-sig")
cleaned.head(1000).to_csv(
    OUTPUT_DIR / "rides_clean_sample1000.csv", index=False, encoding="utf-8-sig"
)

summary = (
    cleaned.groupby("member_casual")["duration_sec"]
    .agg(["count", "mean", "median"])
    .round(1)
)
print(summary.to_string())

report = [
    "共享单车 2024-01 清洗报告",
    f"原始行数: {len(pd.read_csv(RAW_CSV, dtype=str, keep_default_na=False))}",
    f"去重后行数: {len(df)}",
    f"异常标记行数: {int(df['is_anomaly'].sum())}",
    f"剔除硬异常后行数: {len(cleaned)}",
]
(OUTPUT_DIR / "cleaning_report.txt").write_text("\n".join(report) + "\n", encoding="utf-8")
print("\n".join(report))


## 基础聚合统计

在清洗结果上派生日期、小时、星期和时段：

- 凌晨 0–5 点
- 早高峰 6–9 点
- 平峰 10–16 点
- 晚高峰 17–19 点
- 夜间 20–23 点


In [ ]:
cleaned["date"] = cleaned["started_at"].dt.normalize()
cleaned["hour"] = cleaned["started_at"].dt.hour

weekdays = ["周一", "周二", "周三", "周四", "周五", "周六", "周日"]
cleaned["weekday"] = pd.Categorical(
    cleaned["started_at"].dt.day_of_week.map(dict(enumerate(weekdays))),
    categories=weekdays,
    ordered=True,
)

periods = ["凌晨", "早高峰", "平峰", "晚高峰", "夜间"]
cleaned["period"] = pd.Categorical(
    pd.cut(cleaned["hour"], bins=[-1, 5, 9, 16, 19, 23], labels=periods),
    categories=periods,
    ordered=True,
)

cleaned["member_trips"] = cleaned["member_casual"].eq("member").astype(int)
cleaned["casual_trips"] = cleaned["member_casual"].eq("casual").astype(int)
cleaned["electric_trips"] = cleaned["rideable_type"].eq("electric_bike").astype(int)
cleaned["classic_trips"] = cleaned["rideable_type"].eq("classic_bike").astype(int)


def aggregate(key):
    grouped = cleaned.groupby(key, observed=True)
    table = pd.DataFrame(
        {
            "trips": grouped.size(),
            "member_trips": grouped["member_trips"].sum(),
            "casual_trips": grouped["casual_trips"].sum(),
            "electric_trips": grouped["electric_trips"].sum(),
            "classic_trips": grouped["classic_trips"].sum(),
            "avg_duration_sec": grouped["duration_sec"].mean().round(1),
        }
    ).reset_index()
    table["share"] = (table["trips"] / len(cleaned)).round(4)
    return table


by_period = aggregate("period")
by_weekday = aggregate("weekday")
by_hour = aggregate("hour")
by_period


In [ ]:
by_weekday

### 按日期的骑行量时序

In [ ]:
daily = aggregate("date")
full_dates = pd.date_range(
    cleaned["date"].min(), cleaned["date"].max(), freq="D", name="date"
)
daily = daily.set_index("date").reindex(full_dates)
for col in ["trips", "member_trips", "casual_trips", "electric_trips", "classic_trips"]:
    daily[col] = daily[col].fillna(0).astype(int)
daily["share"] = (daily["trips"] / len(cleaned)).round(4)
daily = daily.reset_index()
daily.head(10)


## 保存统计结果

In [ ]:
tables = {
    "stats_by_period.csv": by_period,
    "stats_by_weekday.csv": by_weekday,
    "stats_by_hour.csv": by_hour,
    "stats_daily.csv": daily,
}
for name, table in tables.items():
    table.to_csv(OUTPUT_DIR / name, index=False, encoding="utf-8-sig")

peak_day = daily.loc[daily["trips"].idxmax()]
peak_hour = by_hour.loc[by_hour["trips"].idxmax()]
peak_weekday = by_weekday.loc[by_weekday["trips"].idxmax()]
peak_period = by_period.loc[by_period["trips"].idxmax()]

report = [
    "共享单车 2024-01 聚合统计报告",
    f"统计记录数: {len(cleaned)}",
    f"统计区间: {cleaned['date'].min():%Y-%m-%d} ~ {cleaned['date'].max():%Y-%m-%d}",
    f"日均骑行量: {daily['trips'].mean():.1f}",
    f"单日最高: {peak_day['date']:%Y-%m-%d}，{int(peak_day['trips'])} 次",
    f"最忙时段: {peak_period['period']}，{int(peak_period['trips'])} 次（占 {peak_period['share']:.1%}）",
    f"最忙小时: {int(peak_hour['hour'])} 点，{int(peak_hour['trips'])} 次",
    f"最忙星期: {peak_weekday['weekday']}，{int(peak_weekday['trips'])} 次（占 {peak_weekday['share']:.1%}）",
]
(OUTPUT_DIR / "aggregation_report.txt").write_text("\n".join(report) + "\n", encoding="utf-8")
print("\n".join(report))


## 结论

- 清洗后保留 238,471 条有效骑行记录，时间范围为 2024-01-01 至 2024-01-31
- 日均骑行量约 7,692.6 次，1 月 26 日最高，为 13,977 次
- 平峰（10–16 点）骑行量最大，占 42.2%；17 点为单小时高峰
- 周三骑行量最高，共 45,793 次，占 19.2%
